## Import

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"   # single-threaded so parallel jobs do not fight over cores

import json, os, time, pickle, warnings
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, log_loss, average_precision_score)
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

/Users/waroonragwongsiri/Playground/IntroMLHomework/Data/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup

In [2]:
SEED = 42
N_TRIALS = 30   # same Optuna budget for every model
INNER_K = 3
K, REPEATS = 5, 3
os.makedirs("weights_V6_1", exist_ok=True)

In [3]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df = df[df["Income"] != 666666].copy()  # drop $666,666 income outlier (same treatment as Year_Birth sentinels)
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
print("cleaned:", df.shape, "| Response rate:", round(df["Response"].mean(), 4))

CLF_NUMERIC = ["AcceptedCmp5","AcceptedCmp3","MntWines","MntMeatProducts","NumCatalogPurchases",
    "Recency","Customer_Tenure_Days","MntGoldProds","Income","MntFruits","AcceptedCmp1",
    "AcceptedCmp2","NumWebPurchases","MntSweetProducts","MntFishProducts","NumStorePurchases","Age"]
X_num = df[CLF_NUMERIC].copy()
X_cat = pd.get_dummies(df[["Marital_Status"]], dtype=float)
X_full = pd.concat([X_num, X_cat], axis=1)
FEATURE_COLS = list(X_full.columns)
y_full = df["Response"].astype(float).values
print(f"features: {len(CLF_NUMERIC)} numeric + {X_cat.shape[1]} dummies = {len(FEATURE_COLS)} dims")

cleaned: (2236, 28) | Response rate: 0.1494
features: 17 numeric + 5 dummies = 22 dims


In [4]:
X_all = X_full.values
y_all = y_full.astype(int)
inc_idx = FEATURE_COLS.index("Income")

def prep_split(tr_idx, te_idx):
    """Median-fill Income and standardise using the training rows only."""
    Xtr, Xte = X_all[tr_idx].copy(), X_all[te_idx].copy()
    med = np.nanmedian(Xtr[:, inc_idx])
    for a in (Xtr, Xte):
        a[:, inc_idx] = np.where(np.isnan(a[:, inc_idx]), med, a[:, inc_idx])
    sc = StandardScaler().fit(Xtr)
    return sc.transform(Xtr), sc.transform(Xte), y_all[tr_idx], y_all[te_idx]

## Model

In [5]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier,
                              HistGradientBoostingClassifier, StackingClassifier)
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.neural_network import MLPClassifier

class ClusterRate(BaseEstimator, ClassifierMixin):
    """Unsupervised clustering used as a classifier: each cluster scores the positive rate of its training rows."""
    def __init__(self, method="kmeans", k=8, linkage="ward", seed=0):
        self.method, self.k, self.linkage, self.seed = method, k, linkage, seed
    def fit(self, X, y):
        if self.method == "kmeans":
            self.km_ = KMeans(self.k, n_init=5, random_state=self.seed).fit(X)
            lab, self.centers_ = self.km_.labels_, self.km_.cluster_centers_
        else:
            lab = AgglomerativeClustering(n_clusters=self.k, linkage=self.linkage).fit_predict(X)
            self.centers_ = np.vstack([X[lab == c].mean(0) for c in range(self.k)])
        self.rate_ = np.array([(y[lab == c].sum() + y.mean()) / ((lab == c).sum() + 1) for c in range(self.k)])
        return self
    def predict_proba(self, X):
        d = ((X[:, None, :] - self.centers_[None]) ** 2).sum(-1)
        r = self.rate_[d.argmin(1)]
        return np.c_[1 - r, r]

def _cw(t): return t.suggest_categorical("class_weight", [None, "balanced"])

SPECS = {
 "Logistic regression": dict(tier="curriculum", kind="proba",
    space=lambda t: dict(C=t.suggest_float("C", 1e-3, 1e2, log=True), cw=_cw(t)),
    make=lambda p, s: LogisticRegression(C=p["C"], class_weight=p["cw"], max_iter=3000)),
 "Decision tree": dict(tier="curriculum", kind="proba",
    space=lambda t: dict(d=t.suggest_int("d", 2, 12), leaf=t.suggest_int("leaf", 1, 30), cw=_cw(t)),
    make=lambda p, s: DecisionTreeClassifier(max_depth=p["d"], min_samples_leaf=p["leaf"], class_weight=p["cw"], random_state=s)),
 "Random forest": dict(tier="curriculum", kind="proba",
    space=lambda t: dict(n=t.suggest_int("n", 100, 400, step=100), d=t.suggest_int("d", 3, 16),
                         leaf=t.suggest_int("leaf", 1, 10), mf=t.suggest_categorical("mf", ["sqrt", 0.5]),
                         cw=t.suggest_categorical("cw", [None, "balanced_subsample"])),
    make=lambda p, s: RandomForestClassifier(n_estimators=p["n"], max_depth=p["d"], min_samples_leaf=p["leaf"],
                         max_features=p["mf"], class_weight=p["cw"], n_jobs=1, random_state=s)),
 "ExtraTrees": dict(tier="curriculum", kind="proba",
    space=lambda t: dict(n=t.suggest_int("n", 100, 400, step=100), d=t.suggest_int("d", 3, 16),
                         leaf=t.suggest_int("leaf", 1, 10), mf=t.suggest_categorical("mf", ["sqrt", 0.5]),
                         cw=t.suggest_categorical("cw", [None, "balanced_subsample"])),
    make=lambda p, s: ExtraTreesClassifier(n_estimators=p["n"], max_depth=p["d"], min_samples_leaf=p["leaf"],
                         max_features=p["mf"], class_weight=p["cw"], n_jobs=1, random_state=s)),
 "Gradient boosting": dict(tier="curriculum", kind="proba",
    space=lambda t: dict(n=t.suggest_int("n", 50, 300, step=50), lr=t.suggest_float("lr", 0.01, 0.3, log=True),
                         d=t.suggest_int("d", 2, 5), sub=t.suggest_float("sub", 0.6, 1.0)),
    make=lambda p, s: GradientBoostingClassifier(n_estimators=p["n"], learning_rate=p["lr"], max_depth=p["d"],
                         subsample=p["sub"], random_state=s)),
 "HistGradientBoosting": dict(tier="curriculum", kind="proba",
    space=lambda t: dict(it=t.suggest_int("it", 50, 400, step=50), lr=t.suggest_float("lr", 0.01, 0.3, log=True),
                         leaf=t.suggest_int("leaf", 8, 64), l2=t.suggest_float("l2", 1e-3, 10, log=True), cw=_cw(t)),
    make=lambda p, s: HistGradientBoostingClassifier(max_iter=p["it"], learning_rate=p["lr"], max_leaf_nodes=p["leaf"],
                         l2_regularization=p["l2"], class_weight=p["cw"], random_state=s)),
 "Stacking": dict(tier="curriculum", kind="proba", trials=10,   # only 2 tunable settings, so 10 trials are enough
    space=lambda t: dict(C=t.suggest_float("C", 1e-2, 1e1, log=True), pt=t.suggest_categorical("pt", [False, True])),
    make=lambda p, s: StackingClassifier(
        estimators=[("rf", RandomForestClassifier(100, max_depth=8, class_weight="balanced_subsample", n_jobs=1, random_state=s)),
                    ("gb", GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=s)),
                    ("svm", SVC(C=1.0, probability=True, class_weight="balanced", random_state=s))],
        final_estimator=LogisticRegression(C=p["C"], max_iter=3000), passthrough=p["pt"], cv=3)),
 "Naive Bayes": dict(tier="curriculum", kind="proba",
    space=lambda t: dict(v=t.suggest_float("v", 1e-9, 1e-1, log=True)),
    make=lambda p, s: GaussianNB(var_smoothing=p["v"])),
 "SVM (RBF)": dict(tier="curriculum", kind="decision",
    space=lambda t: dict(C=t.suggest_float("C", 1e-2, 1e2, log=True), g=t.suggest_float("g", 1e-4, 1.0, log=True), cw=_cw(t)),
    make=lambda p, s: SVC(C=p["C"], gamma=p["g"], class_weight=p["cw"])),
 "PCA + Random forest": dict(tier="curriculum", kind="proba",
    space=lambda t: dict(nc=t.suggest_int("nc", 4, 20), n=t.suggest_int("n", 100, 300, step=100),
                         d=t.suggest_int("d", 3, 14), leaf=t.suggest_int("leaf", 1, 10)),
    make=lambda p, s: make_pipeline(PCA(p["nc"], random_state=s),
        RandomForestClassifier(n_estimators=p["n"], max_depth=p["d"], min_samples_leaf=p["leaf"],
                               class_weight="balanced_subsample", n_jobs=1, random_state=s))),
 "PCA + SVM": dict(tier="curriculum", kind="decision",
    space=lambda t: dict(nc=t.suggest_int("nc", 4, 20), C=t.suggest_float("C", 1e-2, 1e2, log=True),
                         g=t.suggest_float("g", 1e-4, 1.0, log=True)),
    make=lambda p, s: make_pipeline(PCA(p["nc"], random_state=s),
        SVC(C=p["C"], gamma=p["g"], class_weight="balanced"))),
 "kNN": dict(tier="curriculum", kind="proba",
    space=lambda t: dict(k=t.suggest_int("k", 3, 51, step=2), w=t.suggest_categorical("w", ["uniform", "distance"]),
                         p=t.suggest_int("p", 1, 2)),
    make=lambda p, s: KNeighborsClassifier(n_neighbors=p["k"], weights=p["w"], p=p["p"])),
 "k-Means (cluster rate)": dict(tier="curriculum", kind="proba",
    space=lambda t: dict(k=t.suggest_int("k", 2, 40)),
    make=lambda p, s: ClusterRate("kmeans", p["k"], seed=s)),
 "Agglomerative (cluster rate)": dict(tier="curriculum", kind="proba",
    space=lambda t: dict(k=t.suggest_int("k", 2, 40), l=t.suggest_categorical("l", ["ward", "average", "complete"])),
    make=lambda p, s: ClusterRate("agglo", p["k"], linkage=p["l"], seed=s)),
 "Perceptron (SLP)": dict(tier="curriculum", kind="decision",
    space=lambda t: dict(a=t.suggest_float("a", 1e-6, 1e-1, log=True), pen=t.suggest_categorical("pen", [None, "l2"]), cw=_cw(t)),
    make=lambda p, s: Perceptron(alpha=p["a"], penalty=p["pen"], class_weight=p["cw"], max_iter=2000, random_state=s)),
 "MLP (sklearn)": dict(tier="curriculum", kind="proba",
    space=lambda t: dict(h1=t.suggest_int("h1", 16, 128, step=16), two=t.suggest_categorical("two", [False, True]),
                         a=t.suggest_float("a", 1e-5, 1e-1, log=True), lr=t.suggest_float("lr", 1e-3, 1e-2, log=True)),
    make=lambda p, s: MLPClassifier(hidden_layer_sizes=(p["h1"], p["h1"] // 2) if p["two"] else (p["h1"],),
                         alpha=p["a"], learning_rate_init=p["lr"], max_iter=400, early_stopping=True,
                         n_iter_no_change=15, random_state=s)),
}

## Sanity Check

In [6]:
from sklearn.dummy import DummyClassifier
a, b = next(StratifiedKFold(K, shuffle=True, random_state=SEED).split(X_all, y_all))
assert len(set(a) & set(b)) == 0, "train / eval rows overlap"
Xa, Xb, ya, yb = prep_split(a, b)
d = DummyClassifier(strategy="most_frequent").fit(Xa, ya)
pred = d.predict(Xb)
print(f"all-'no' baseline: accuracy {accuracy_score(yb, pred):.4f} | F1 {f1_score(yb, pred, zero_division=0):.4f} | positive rate {yb.mean():.4f}")
assert f1_score(yb, pred, zero_division=0) == 0.0

all-'no' baseline: accuracy 0.8504 | F1 0.0000 | positive rate 0.1496


## Training

In [7]:
def score_of(est, X, kind):
    return est.predict_proba(X)[:, 1] if kind == "proba" else est.decision_function(X)

def inner_oof(spec, params, X, y, seed):
    """Out-of-fold scores on the training rows (inner K-fold)."""
    oof = np.zeros(len(y))
    for a, b in StratifiedKFold(INNER_K, shuffle=True, random_state=seed).split(X, y):
        est = spec["make"](params, seed)
        est.fit(X[a], y[a])
        oof[b] = score_of(est, X[b], spec["kind"])
    return oof

def tune(spec, X, y, seed):
    """Optuna (TPE) maximising average precision of the inner out-of-fold scores."""
    def objective(trial):
        return average_precision_score(y, inner_oof(spec, spec["space"](trial), X, y, seed))
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=spec.get("trials", N_TRIALS))
    return spec["space"](optuna.trial.FixedTrial(study.best_params))

def best_threshold(y, s):
    """F1-maximising threshold, searched on training-side out-of-fold scores only."""
    cand = np.unique(np.quantile(s, np.linspace(0.40, 0.99, 120)))
    f = [f1_score(y, (s >= t).astype(int), zero_division=0) for t in cand]
    return float(cand[int(np.argmax(f))])

def fit_eval(spec, params, Xa, ya, Xb, yb, seed):
    """Threshold from inner OOF on (Xa, ya); refit on all of (Xa, ya); score (Xb, yb)."""
    thr = best_threshold(ya, inner_oof(spec, params, Xa, ya, seed))
    est = spec["make"](params, seed)
    est.fit(Xa, ya)
    s = score_of(est, Xb, spec["kind"])
    pred = (s >= thr).astype(int)
    row = {"f1": f1_score(yb, pred, zero_division=0), "accuracy": accuracy_score(yb, pred),
           "precision": precision_score(yb, pred, zero_division=0),
           "recall": recall_score(yb, pred, zero_division=0),
           "roc_auc": roc_auc_score(yb, s), "threshold": thr}
    if spec["kind"] == "proba":
        row["bce"] = log_loss(yb, np.clip(s, 1e-12, 1 - 1e-12))
    return row, s

In [8]:
cache = "weights_V6_1/_progress_classification_sklearn.pkl"
state = pickle.load(open(cache, "rb")) if os.path.exists(cache) else {"rows": [], "scores": {}}
done = {(r["model"], r["repeat"], r["fold"]) for r in state["rows"]}
for rep in range(1, REPEATS + 1):
    seed_r = SEED + rep - 1               # repeat 1 uses the same folds as V1_3 / V3_1
    for fold, (a, b) in enumerate(StratifiedKFold(K, shuffle=True, random_state=seed_r).split(X_all, y_all), start=1):
        Xa, Xb, ya, yb = prep_split(a, b)
        for name, spec in SPECS.items():
            if (name, rep, fold) in done:
                continue
            t0 = time.time()
            params = tune(spec, Xa, ya, seed_r * 100 + fold)
            row, s = fit_eval(spec, params, Xa, ya, Xb, yb, seed_r * 100 + fold)
            state["rows"].append(dict(model=name, tier=spec["tier"], selection="inner validation", repeat=rep, fold=fold,
                                      n_test=int(len(yb)), **{k: round(float(v), 4) for k, v in row.items()},
                                      params=params, seconds=round(time.time() - t0, 1)))
            key = f"{name}|r{rep}"
            state["scores"].setdefault(key, np.full(len(y_all if "clf" == "clf" else y_raw), np.nan))[b] = s
        pickle.dump(state, open(cache, "wb"))
        print(f"repeat {rep} fold {fold} done | rows so far {len(state['rows'])}", flush=True)

repeat 1 fold 1 done | rows so far 16


repeat 1 fold 2 done | rows so far 32


repeat 1 fold 3 done | rows so far 48


repeat 1 fold 4 done | rows so far 64


repeat 1 fold 5 done | rows so far 80


...

repeat 3 fold 1 done | rows so far 176


repeat 3 fold 2 done | rows so far 192


repeat 3 fold 3 done | rows so far 208


repeat 3 fold 4 done | rows so far 224


repeat 3 fold 5 done | rows so far 240


## K-Fold Summary

In [9]:
kdf = pd.DataFrame(state["rows"])
KEYS = ['f1', 'accuracy', 'precision', 'recall', 'roc_auc', 'bce']
summary = {m: {k: {"mean": round(float(g[k].mean()), 4), "std": round(float(g[k].std()), 4)} for k in KEYS if k in g and g[k].notna().any()}
           for m, g in kdf.groupby("model")}
rank = kdf.groupby("model")["f1"].agg(["mean", "std"]).sort_values("mean", ascending=False).round(4)
print(rank.to_string())
json.dump({"k": K, "repeats": REPEATS, "n_trials": N_TRIALS, "folds": state["rows"], "summary": summary},
          open("weights_V6_1/classification_sklearn_kfold_metrics.json", "w"), indent=2, default=str)
np.savez("weights_V6_1/classification_sklearn_kfold_scores.npz", **state["scores"])

                                mean     std
model                                       
HistGradientBoosting          0.5784  0.0322
Stacking                      0.5742  0.0541
Gradient boosting             0.5671  0.0497
Random forest                 0.5648  0.0508
PCA + Random forest           0.5618  0.0347
SVM (RBF)                     0.5583  0.0499
MLP (sklearn)                 0.5562  0.0655
ExtraTrees                    0.5503  0.0551
PCA + SVM                     0.5497  0.0339
Logistic regression           0.5395  0.0575
kNN                           0.5359  0.0520
Naive Bayes                   0.4778  0.0376
k-Means (cluster rate)        0.4724  0.0387
Agglomerative (cluster rate)  0.4648  0.0368
Perceptron (SLP)              0.4564  0.0661
Decision tree                 0.4437  0.0741
